# Kernels: the two mean maps and the shared schedule

`specdiff/kernels.py` is the whole model interface. Appendix C, eq. (24) assumes that the
proposal and the target are isotropic Gaussians that **share a variance schedule and differ
only in their means**:

```
P_n(. | y) = N(m^p_n(y), sigma_n^2 I)      the cheap draft kernel
Q_n(. | y) = N(m^q_n(y), sigma_n^2 I)      the expensive one we want to sample from
```

So a model is three objects:

| you supply | base class | called |
| --- | --- | --- |
| `m^q` | `TargetTransition` | once per round, batched over the tree's internal nodes |
| `m^p` | `ProposalTransition` | once per tree *level* while drafting |
| `{sigma_n}` | `NoiseSchedule` | per step, by the sampler |

The sampler never learns what is behind them — a denoiser, a distilled draft network, an
analytic score, or the paper's delayed reverse drift.

In [1]:
import numpy as np

from specdiff import (
    ConstantSchedule,
    DelayedDriftProposal,
    IdentityProposal,
    MirrorProposal,
    NoiseSchedule,
    ProposalTransition,
    TabulatedSchedule,
    TargetTransition,
)

rng = np.random.default_rng(0)

## 1. `NoiseSchedule`

Subclasses implement `sigma(step)`. The sampler calls the *instance* — `schedule(n)` — and
`__call__` refuses a non-positive scale, because at zero churn both kernels collapse to point
masses, their total-variation distance is 1, and speculation is vacuous (Remark 3).

In [2]:
class SqrtSchedule(NoiseSchedule):
    """sigma_n decaying to a floor as the trajectory denoises."""

    def __init__(self, num_steps, scale=0.4, floor=0.05):
        self.num_steps = int(num_steps)
        self.scale = float(scale)
        self.floor = float(floor)

    def sigma(self, step):
        return self.floor + self.scale * np.sqrt((self.num_steps - step) / self.num_steps)


N = 20
schedule = SqrtSchedule(N)
[round(schedule(n), 4) for n in (0, 1, N // 2, N - 1)]

[0.45, 0.4399, 0.3328, 0.1394]

In [3]:
# The two batteries-included schedules.
constant = ConstantSchedule(0.3)
tabulated = TabulatedSchedule([schedule(n) for n in range(N)])

print("constant     ", constant(0), constant(19))
print("tabulated    ", round(tabulated(0), 4), round(tabulated(19), 4), "len:", len(tabulated))

# The guard. Note it fires on __call__, not on sigma(), so a schedule is free to
# *store* a zero -- it just cannot be used at a step the sampler visits.
zero = ConstantSchedule(0.0)
print("sigma(0)     ", zero.sigma(0), "<- raw value, no check")
try:
    zero(0)
except ValueError as exc:
    print("schedule(0)  ", type(exc).__name__, "->", str(exc).splitlines()[0])

constant      0.3 0.3
tabulated     0.45 0.1394 len: 20
sigma(0)      0.0 <- raw value, no check
schedule(0)   ValueError -> sigma(0) = 0.0: speculation is vacuous at zero churn (the transitions become point masses, TV distance 1). See Remark 3 in the paper.


## 2. `TargetTransition` — `m^q`, and where the NFEs are counted

You override `means(states, steps)`:

* `states` is a stack of shape `(rows, *state_shape)` — **every internal node of the round's
  draft tree at once**, not one state;
* `steps` is a tuple of the same length, because nodes at different depths of the tree realise
  different steps `n + |u|`;
* the return must have `rows` rows, and that is checked.

`means` must be a *single* batched evaluation of your network. That is the whole point: one
round costs one forward pass regardless of how many states the tree drafted.

In [4]:
class LinearReverseKernel(TargetTransition):
    """A stand-in for the expensive network: m^q_n(y) = y + gamma_n (c - y).

    Every Euler-Maruyama reverse step has this shape -- one drift evaluation folded
    into a mean. Here the drift is linear, so every number below can be checked by
    hand; in production this is a call to a denoiser.
    """

    def __init__(self, center, gammas):
        super().__init__()                      # sets up the NFE counters
        self.center = np.asarray(center, dtype=float)
        self.gammas = np.asarray(gammas, dtype=float)

    def means(self, states, steps):
        gamma = self.gammas[list(steps)].reshape(-1, *([1] * (states.ndim - 1)))
        return states + gamma * (self.center - states)


dim = 4
target = LinearReverseKernel(center=np.zeros(dim), gammas=np.linspace(0.05, 0.25, N))
states = rng.standard_normal((3, dim))
target(states, (0, 5, 11)).round(3)

array([[ 0.119, -0.125,  0.608,  0.1  ],
       [-0.481,  0.324,  1.17 ,  0.85 ],
       [-0.587, -1.056, -0.52 ,  0.034]])

In [5]:
# __call__ counts, means() does not. Call the instance -- the cost metric the paper
# plots against is defined on these counters.
target.reset_stats()

target(states, (0, 5, 11))          # one batched call, three states
target(states[:1], (7,))            # another call, one state

print("target_calls (NFEs)      ", target.num_calls)
print("states pushed through    ", target.num_states)

target.means(states, (0, 5, 11))    # bypasses the accounting -- don't do this
print("after a raw .means() call", target.num_calls, target.num_states)

target_calls (NFEs)       2
states pushed through     4
after a raw .means() call 2 4


In [6]:
# The row-count check catches the most common modelling bug: a means() that
# collapses the batch (e.g. an accidental mean over rows, or a squeeze).
class BrokenTarget(TargetTransition):
    def means(self, states, steps):
        return states.mean(axis=0, keepdims=True)


try:
    BrokenTarget()(states, (0, 1, 2))
except ValueError as exc:
    print(type(exc).__name__, "->", exc)

ValueError -> BrokenTarget.means returned 1 rows for 3 states


## 3. `ProposalTransition` — `m^p`, and its three lifecycle hooks

The proposal has the same `means` signature but is called once per tree *level* during
drafting, with every parent at that level. It is cheap by assumption, so nothing counts it.

The three hooks exist because the interesting proposals are stateful:

| hook | when | why |
| --- | --- | --- |
| `on_round_start(step, root_state)` | once per round, before drafting | a delayed drift needs to know where the round starts |
| `on_verified(step, state, target_mean)` | per committed node, after its target mean was computed | the channel that makes root-drift prefetching possible |
| `reset()` | at the start of each `sample()` | drop per-run state |

A stateless proposal inherits all three as no-ops. Two are shipped:

In [7]:
y = rng.standard_normal((2, dim))

identity = IdentityProposal()       # m^p(y) = y: the worst useful proposal, and free
mirror = MirrorProposal(target)     # m^p = m^q: a perfect proposal, delta = 0 everywhere

print("identity == y        ", np.allclose(identity.means(y, (3, 3)), y))
print("mirror   == target   ", np.allclose(mirror.means(y, (3, 3)), target(y, (3, 3))))
print("stateful flags       ", identity.stateful, mirror.stateful)

identity == y         True
mirror   == target    True
stateful flags        False False


In [8]:
# MirrorProposal is a *test device*, not a model: it reaches for the target, so every
# drafting level it serves costs a real NFE. Useful for isolating sampler bugs from
# coupling bugs (delta = 0 => everything accepts); useless in production.
target.reset_stats()
mirror.means(y, (3, 3))
print("NFEs spent by one mirror drafting level:", target.num_calls)

NFEs spent by one mirror drafting level: 1


### Writing your own

A distilled draft network is stateless, so it is just a `means`:

In [9]:
class CoarseDrift(ProposalTransition):
    """Same drift as the target but scaled -- a stand-in for a cheap draft network."""

    def __init__(self, center, gammas, quality=0.8):
        self.center = np.asarray(center, dtype=float)
        self.gammas = np.asarray(gammas, dtype=float)
        self.quality = float(quality)

    def means(self, states, steps):
        gamma = self.gammas[list(steps)].reshape(-1, *([1] * (states.ndim - 1)))
        return states + self.quality * gamma * (self.center - states)


coarse = CoarseDrift(target.center, target.gammas)
mu_q = target(y, (10, 10))
mu_p = coarse.means(y, (10, 10))

sigma = schedule(10)
delta = np.linalg.norm(mu_q - mu_p, axis=1) / sigma
print("delta per row:", delta.round(4))

delta per row: [0.2562 0.1199]


`delta = ||m^q - m^p|| / sigma` is *the* number: it is the normalised mean mismatch, and every
acceptance probability in the paper is a function of it alone. A proposal is good exactly when
it keeps `delta` small relative to the noise it is competing with. See the verifier tutorial for
what a rule does with it.

## 4. `DelayedDriftProposal` — the self-speculative proposal (eq. 7)

This is the proposal that needs no second network. The target mean is
`m^q_n(y) = y + gamma b^q(y)`, so the drift increment is recoverable from means alone:

```
gamma b^q_{n'}(Y~) = m^q_{n'}(Y~) - Y~
```

The proposal freezes that increment at the root of the round and reuses it at every depth:

```
m^p(y) = y + (m^q_{n'}(Y~) - Y~)
```

Note what this buys: the proposal never needs access to your drift, your score, or your
network — only to means it has already been handed.

In [10]:
proposal = DelayedDriftProposal(target)     # prefetch=True is the paper's default
target.reset_stats()
proposal.reset()

root = rng.standard_normal(dim)
proposal.on_round_start(0, root)            # the one unavoidable warm-up call, at n = 0
print("warm-up NFEs:", target.num_calls)

# At the root the frozen drift is exact, so m^p = m^q there.
print("exact at the root:", np.allclose(proposal.means(root[None], (0,)), target.means(root[None], (0,))))

# Away from the root it is a frozen-drift approximation -- that gap is delta.
elsewhere = root + 0.5 * rng.standard_normal((3, dim))
gap = proposal.means(elsewhere, (1, 1, 1)) - target.means(elsewhere, (1, 1, 1))
print("delta at depth 1: ", (np.linalg.norm(gap, axis=1) / schedule(1)).round(4))

warm-up NFEs: 1
exact at the root: True
delta at depth 1:  [0.112  0.104  0.0785]


In [11]:
# The hooks in slow motion. This is what the sampler does around a round, by hand.
target.reset_stats()
proposal.reset()

state = root
for n in (0, 1, 2):
    before = target.num_calls
    proposal.on_round_start(n, state)                    # warms up only at n = 0
    warmup = target.num_calls - before

    drafted = proposal.means(state[None], (n,))[0]       # drafting uses the frozen drift
    committed = drafted + schedule(n) * rng.standard_normal(dim)

    mean_at_parent = target(state[None], (n,))[0]        # verification pays for this anyway
    proposal.on_verified(n, state, mean_at_parent)       # ... so hand it back: prefetch

    print(f"round n={n}  NFEs charged to the proposal: {warmup}   total so far: {target.num_calls}")
    state = committed

round n=0  NFEs charged to the proposal: 1   total so far: 2
round n=1  NFEs charged to the proposal: 0   total so far: 3
round n=2  NFEs charged to the proposal: 0   total so far: 4


Only the warm-up cost an extra call: from `n = 1` on, `on_round_start` finds an increment
already in hand — the one verification computed at the previous round's parent. That is
**root-drift prefetching**, and it is why the delayed drift is free per round.

`prefetch=False` gives up that trick and re-evaluates the target at every round's root. It
costs one extra NFE per round but yields a strictly fresher, better proposal, which is what you
want when isolating the effect of proposal quality:

In [12]:
for prefetch in (True, False):
    p = DelayedDriftProposal(target, prefetch=prefetch)
    target.reset_stats()
    p.reset()
    st = root
    for n in range(5):
        p.on_round_start(n, st)
        st = p.means(st[None], (n,))[0] + schedule(n) * rng.standard_normal(dim)
        p.on_verified(n, st, target(st[None], (n,))[0])
    # 5 rounds, each of which pays 1 NFE for verification in this hand-rolled loop
    print(f"prefetch={prefetch!s:5}  NFEs over 5 rounds: {target.num_calls}")

prefetch=True   NFEs over 5 rounds: 6
prefetch=False  NFEs over 5 rounds: 10


### Staleness is the whole cost of the trick

The frozen increment ages as the round descends: the deeper the level, the further the state
has drifted from the one the increment was computed at. `delta` grows with depth, and that is
what caps the useful lookahead `L`.

In [13]:
def staleness_profile(levels=6, trials=400):
    """Mean delta at each depth of a round, averaged over independent roots."""
    proposal = DelayedDriftProposal(target)
    out = np.zeros(levels)
    for _ in range(trials):
        proposal.reset()
        state = rng.standard_normal(dim)
        proposal.on_round_start(0, state)
        for level in range(1, levels + 1):
            step = level - 1
            mu_p = proposal.means(state[None], (step,))[0]
            mu_q = target.means(state[None], (step,))[0]
            out[level - 1] += np.linalg.norm(mu_q - mu_p) / schedule(step)
            state = mu_p + schedule(step) * rng.standard_normal(dim)   # descend one level
    return out / trials


for level, d in enumerate(staleness_profile(), start=1):
    print(f"  level {level}:  mean delta = {d:.4f}   {'#' * int(60 * d)}")

  level 1:  mean delta = 0.0000   
  level 2:  mean delta = 0.1192   #######
  level 3:  mean delta = 0.2017   ############
  level 4:  mean delta = 0.2884   #################
  level 5:  mean delta = 0.3838   #######################
  level 6:  mean delta = 0.4791   ############################


## 5. `stateful`, and why it matters later

`DelayedDriftProposal` sets `stateful = True`. That flag is not decoration: under the batched
sampler a single cached increment belongs to *one* trajectory, and sharing it across a batch
whose rows sit at different steps would silently corrupt every row but one. The batched sampler
refuses rather than allows it.

In [14]:
from specdiff import StatelessBatchedProposal

print("DelayedDriftProposal.stateful:", DelayedDriftProposal(target).stateful)
try:
    StatelessBatchedProposal(DelayedDriftProposal(target))
except TypeError as exc:
    print(type(exc).__name__, "->", str(exc).splitlines()[0])

DelayedDriftProposal.stateful: True
TypeError -> DelayedDriftProposal is stateful and cannot be shared across a batch: its cached state belongs to one trajectory. Use a BatchedProposal (e.g. BatchedDelayedDriftProposal) or wrap per slot with PerSlotProposal.


## Recap

* `NoiseSchedule.sigma(step)` — shared by both kernels by assumption, not by convention;
  `__call__` refuses zero.
* `TargetTransition.means(states, steps)` — one batched network call per round; call the
  *instance* so the NFE counters stay honest.
* `ProposalTransition.means(states, steps)` plus `on_round_start` / `on_verified` / `reset` —
  the hooks that let a stateful proposal live outside the sampler.
* `DelayedDriftProposal` — eq. (7), free per round thanks to root-drift prefetching; its `delta`
  grows with depth, which is what bounds `L`.

**Next:** [`verifier_tutorial.ipynb`](verifier_tutorial.ipynb) — what a rule does with
`delta`, and the contract it has to keep.